In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.5040000000000002, 10: 0.9345000000000002, 20: 0.9470000000000003, 30: 0.9490000000000004, 40: 0.9530000000000001, 50: 0.9470000000000003, 60: 0.9390000000000003, 70: 0.9465, 80: 0.9455, 90: 0.9445, 100: 0.9480000000000001, 110: 0.9530000000000001, 120: 0.9525, 130: 0.9455, 140: 0.9475, 150: 0.9445000000000002, 160: 0.9455000000000002, 170: 0.9465000000000001, 180: 0.9490000000000002, 190: 0.9520000000000002, 200: 0.9520000000000003, 210: 0.9489999999999998, 220: 0.9515, 230: 0.9444999999999999, 240: 0.9480000000000001, 250: 0.9490000000000001, 260: 0.9475, 270: 0.958421052631579, 280: 0.9515789473684213, 290: 0.9447368421052635, 300: 0.9436842105263158}
{0: 0.056284, 10: 0.000999749999999999, 20: 0.0010309999999999991, 30: 0.0008589999999999993, 40: 0.0007309999999999996, 50: 0.0007109999999999992, 60: 0.0013589999999999993, 70: 0.001107749999999999, 80: 0.0014597499999999992, 90: 0.001129749999999999, 100: 0.000975999999999999, 110: 0.0008109999999999994, 120: 0.0008337499999999

In [4]:
mean_1_10, var_1_10 = summarize_by_step(root='.', start=1,  end=10)
mean_11_20, var_11_20 = summarize_by_step(root='.', start=11, end=20)
mean_21_30, var_21_30 = summarize_by_step(root='.', start=21, end=30)
mean_31_40, var_31_40 = summarize_by_step(root='.', start=31, end=40)

In [5]:
print(mean_1_10)
print(var_1_10)

{0: 0.638, 10: 0.9359999999999999, 20: 0.954, 30: 0.95, 40: 0.952, 50: 0.944, 60: 0.938, 70: 0.9359999999999999, 80: 0.95, 90: 0.9480000000000001, 100: 0.95, 110: 0.9480000000000001, 120: 0.95, 130: 0.9339999999999999, 140: 0.95, 150: 0.9359999999999999, 160: 0.9559999999999998, 170: 0.9440000000000002, 180: 0.95, 190: 0.952, 200: 0.958, 210: 0.95, 220: 0.9400000000000002, 230: 0.9339999999999998, 240: 0.9540000000000001, 250: 0.9480000000000001, 260: 0.952, 270: 0.95, 280: 0.9480000000000001, 290: 0.938, 300: 0.9380000000000001}
{0: 0.0018759999999999996, 10: 0.0007039999999999985, 20: 0.0006439999999999999, 30: 0.0006599999999999995, 40: 0.0004959999999999997, 50: 0.0007039999999999992, 60: 0.000595999999999999, 70: 0.0008639999999999988, 80: 0.0008999999999999994, 90: 0.001215999999999999, 100: 0.0008999999999999994, 110: 0.0004959999999999993, 120: 0.0004999999999999991, 130: 0.0004039999999999987, 140: 0.0006599999999999988, 150: 0.0005439999999999986, 160: 0.000623999999999999, 1

In [6]:
print(mean_11_20)
print(var_11_20)

{0: 0.21400000000000002, 10: 0.9220000000000003, 20: 0.9359999999999999, 30: 0.938, 40: 0.946, 50: 0.944, 60: 0.9299999999999999, 70: 0.9359999999999999, 80: 0.9280000000000002, 90: 0.932, 100: 0.9259999999999999, 110: 0.9460000000000001, 120: 0.9400000000000001, 130: 0.9399999999999998, 140: 0.93, 150: 0.9339999999999999, 160: 0.9299999999999999, 170: 0.9319999999999998, 180: 0.9400000000000001, 190: 0.942, 200: 0.942, 210: 0.9380000000000001, 220: 0.942, 230: 0.93, 240: 0.9299999999999999, 250: 0.938, 260: 0.9419999999999998, 270: 0.951111111111111, 280: 0.9422222222222223, 290: 0.9377777777777779, 300: 0.9311111111111112}
{0: 0.005044000000000001, 10: 0.001635999999999999, 20: 0.0010239999999999987, 30: 0.0006759999999999992, 40: 0.001123999999999999, 50: 0.0009439999999999987, 60: 0.0012999999999999989, 70: 0.0013439999999999988, 80: 0.001775999999999999, 90: 0.0011359999999999988, 100: 0.001203999999999999, 110: 0.0012039999999999991, 120: 0.000959999999999999, 130: 0.001439999999

In [7]:
print(mean_21_30)
print(var_21_30)

{0: 0.8, 10: 0.9399999999999998, 20: 0.95, 30: 0.944, 40: 0.9620000000000001, 50: 0.9460000000000001, 60: 0.954, 70: 0.9599999999999997, 80: 0.9460000000000001, 90: 0.954, 100: 0.9559999999999998, 110: 0.958, 120: 0.954, 130: 0.9559999999999998, 140: 0.96, 150: 0.944, 160: 0.9479999999999998, 170: 0.954, 180: 0.9579999999999999, 190: 0.954, 200: 0.9560000000000001, 210: 0.958, 220: 0.9640000000000001, 230: 0.9480000000000001, 240: 0.9559999999999998, 250: 0.954, 260: 0.9499999999999998, 270: 0.958, 280: 0.95, 290: 0.9559999999999998, 300: 0.9440000000000002}
{0: 0.005279999999999997, 10: 0.00039999999999999937, 20: 0.0008199999999999988, 30: 0.0009439999999999992, 40: 0.00043599999999999986, 50: 0.0003239999999999994, 60: 0.0006439999999999996, 70: 0.0007199999999999999, 80: 0.0010439999999999991, 90: 0.0008039999999999995, 100: 0.00030399999999999975, 110: 0.0005160000000000001, 120: 0.0007239999999999994, 130: 0.0006239999999999999, 140: 0.00047999999999999996, 150: 0.000543999999999

In [8]:
print(mean_31_40)
print(var_31_40)

{0: 0.364, 10: 0.9399999999999998, 20: 0.9480000000000001, 30: 0.9639999999999999, 40: 0.952, 50: 0.9540000000000001, 60: 0.9339999999999999, 70: 0.9540000000000001, 80: 0.9580000000000002, 90: 0.944, 100: 0.9600000000000002, 110: 0.96, 120: 0.966, 130: 0.952, 140: 0.95, 150: 0.9640000000000001, 160: 0.9480000000000002, 170: 0.9560000000000001, 180: 0.9480000000000001, 190: 0.96, 200: 0.952, 210: 0.95, 220: 0.96, 230: 0.966, 240: 0.9520000000000002, 250: 0.9560000000000001, 260: 0.9460000000000003, 270: 0.9755555555555557, 280: 0.9666666666666668, 290: 0.9466666666666667, 300: 0.9622222222222222}
{0: 0.0036639999999999997, 10: 0.0010399999999999993, 20: 0.0014559999999999994, 30: 0.0007839999999999994, 40: 0.0007360000000000001, 50: 0.0008039999999999995, 60: 0.0025640000000000003, 70: 0.0010439999999999987, 80: 0.001635999999999999, 90: 0.001103999999999999, 100: 0.0007999999999999992, 110: 0.0008799999999999994, 120: 0.0008039999999999999, 130: 0.0008159999999999996, 140: 0.001459999